mix them pixels yooo

In [2]:
import os 
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from math import isnan
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.metrics import r2_score, confusion_matrix
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
import linear_mixing as lmx
from scipy.optimize import curve_fit
from sklearn.cluster import KMeans
import randomforest_CHAIN_S2 as rfcc

In [3]:
# Magic function to auto-update imported first-party scripts 
%load_ext autoreload
%autoreload 2

In [97]:
# Plotting settings and functions

plt.rcParams['font.family'] = 'serif' #sans-serif'

def plot_multi_result(x_array, 
                      y_array, 
                      alpha, 
                      colour,
                      metric, 
                      cmap, 
                      fig_title, 
                      x_axis_label, 
                      y_axis_label, 
                      stats_values, 
                      line_1_1, 
                      sensor_code, 
                      class_code,
                      h_lines = None, 
                      save = False, 
                      output_dir = None,
                      output_name = None):
    fig, ax = plt.subplots(x_array.shape[1]//2 + x_array.shape[1]%2, 2, figsize = (8,8),) #sharex = True, sharey = True)
    fig.tight_layout(pad = 3)
    if isinstance(y_array, pd.DataFrame):
        y_array = y_array.values
    if isinstance(stats_values, pd.DataFrame):
        stats_values = stats_values.values

    for col in range(x_array.shape[1]):
        col_name = x_array.columns[col]
        ax[col//2, col%2].scatter(x_array[col_name], y_array[:,col], alpha = alpha, c = colour, cmap = cmap)
        ax[col//2, col%2].set_xlabel(f"{x_axis_label} of {col_name.replace("_", " ")}")
        ax[col//2, col%2].set_ylabel (y_axis_label)
        if stats_values is not None:
            ax[col//2, col%2].annotate(f"{metric} = {stats_values[metric][col]:.3f}", xy = (0.6, 0.1), xycoords = "axes fraction")
        if line_1_1:
            ax[col//2, col%2].axline((0,0), slope = 1, color = "black", linestyle = "--")
        if h_lines is not None:
            for h in h_lines:
                ax[col//2, col%2].axhline(y = h[0], color = h[1], linestyle = ":")
    if x_array.shape[1] % 2 == 1:
        ax[-1, -1].set_visible(False)  
    fig.suptitle(fig_title, y = 1)
    
    if save: 
        if not sensor_code or not class_code:
            ValueError("Sensor code and class code must be provided to save the figure.")
        if output_dir is None :
            output_dir = "./"
        
        os.makedirs(output_dir, exist_ok=True)
        filename = f"{sensor_code}_{class_code}_{output_name}.svg"
        filepath = os.path.join(output_dir, filename)
        fig.savefig(filepath, bbox_inches = "tight")
    plt.show()

def calc_eval_stats(true_values, predicted):
    r2_list = []
    mse_list = []
    rmse_list = []
    mae_list = []
    mape_list = []
    all_stats = {}
    predicted = pd.DataFrame(predicted)
    true_values = pd.DataFrame(true_values)
    
    for i in range(true_values.shape[1]):
        r2 = r2_score(true_values.iloc[:, i], predicted.iloc[:, i])
        mse = mean_squared_error(true_values.iloc[:, i], predicted.iloc[:, i])
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(true_values.iloc[:, i], predicted.iloc[:, i])
        # mape = np.mean(np.abs((true_values.iloc[:, i] - predicted.iloc[:, i]) / true_values.iloc[:, i]) * 100)
        # # den = true_values.iloc[:, 1]
        # mask = den != 0
        # mape = np.abs(den[mask] - predicted[mask, i])/den[mask] *100
        # mape = np.mean(mape)
        
        den = true_values.iloc[:, i].replace(0, np.nan)

        mape = np.nanmean(np.abs(true_values.iloc[:, i] - predicted.iloc[:, i]) / den * 100)

        
        
        # try: 
        #     mape = np.abs(true_values.iloc[:, i] - predicted[:, i]) / true_values.iloc[:, i]*100
        # except ZeroDivisionError:
        #     mape = np.nan
        # clean_mape = [x for x in mape if not isnan(x)]
        # mape = np.mean(clean_mape)
        
        
        r2_list.append(r2)
        mse_list.append(mse)
        rmse_list.append(rmse)
        mae_list.append(mae)
        mape_list.append(mape)
        
    all_stats["R2"] = r2_list
    all_stats["MSE"] = mse_list
    all_stats["RMSE"] = rmse_list
    all_stats['MAE'] = mae_list
    all_stats['MAPE'] = mape_list
    
    return all_stats

def sigmoid(x, L ,x0, k, b):
    y = L / (1 + np.exp(-k*(x-x0))) + b
    return (y)

def fit_best_curve(x, y, mat, sensor_code, class_code, plot = True, criterion = "R2", colours= dict()):
     
    """Fit a set of lines to the data and decide on the best one based on R squared value.
        
    Args
        x (array-like): Independent variable data.
        y (array-like): Dependent variable data.
        mat (str) : Name of class being fitted.
        plot (bool): Whether to plot the best fit curve. Default is True.
        criterion (str): Criterion to evaluate the best fit. Default is "R2". Options are MAE, RMSE, MSE, MAPE, R2.
        
    Output
        best_fit_params (tuple): Parameters of the best fit curve.
        all_curve_params (dict): Parameters of all fitted curves.
        best_fit_line (str) : Type of curve that fits best
        all_stats (dict) : Evaluation stats of the fitted curves.
    """
    
    all_curve_params = {}
    
    colours = "teal" if None else colours

    # Fit sigmoid curve    
    p0 = [max(y), np.mean(x), 1, 0] # mandatory initial guess
    popt, _ = curve_fit(sigmoid, x, y, p0, method='trf')
    x_linspace = np.linspace(np.min(x), np.max(x), 500)
    y_fitted_sigmoid = sigmoid(x_linspace, *popt)
    y_pred_sigmoid = sigmoid(x, *popt)
    all_curve_params = {'sigmoid': popt}

    sigmoid_stats = calc_eval_stats(y, y_pred_sigmoid)
    all_stats = {'sigmoid' : sigmoid_stats}
    
    # Fit straight line
    b, m = np.polyfit(x, y, 1)
    all_curve_params["linear"] = (b, m)
    y_fitted_linear = b*x_linspace + m
    y_pred_linear = b*x +m

    linear_stats = calc_eval_stats(y, y_pred_linear)
    all_stats['linear'] = linear_stats
    
    # Pick best curve
    if criterion == "R2":
        sigmoid_r2 = sigmoid_stats['R2'][0]
        linear_r2 = linear_stats['R2'][0]
        if sigmoid_r2 > linear_r2:
            best_fit_line = "sigmoid"
            best_fit_params = popt
        else:
            best_fit_line = "linear"
            best_fit_params = (m, b)
    elif criterion == "MAE":
        sigmoid_mae = sigmoid_stats['MAE'][0]
        linear_mae = linear_stats['MAE'][0]
        if sigmoid_mae < linear_mae:
            best_fit_line = "sigmoid"
            best_fit_params = popt
        else:
            best_fit_line = "linear"
            best_fit_params = (m, b)
    
    # Optional plotting
    if plot: 
        ax = plt.axes()
        ax.scatter(x, y, color = colours, alpha = 0.5,)
        ax.plot(x_linspace, y_fitted_sigmoid, '--', color = "red", label='Fitted curve - Sigmoid')
        ax.plot(x_linspace, y_fitted_linear, '--', color = "gold", label='Fitted curve - Linear')
        ax.axline((0,0), slope = 1, color = "black", linestyle = "dotted", label = "1:1 line")
        ax.set_title(f'Fitting curves to fit {mat} prediction data')
        ax.set_ylabel(f'Predicted {mat} probability')
        ax.set_ylim(0, 1)
        ax.set_xlabel(f'{mat} fractional cover')
        ax.annotate(f" Best fit: {best_fit_line.capitalize()} \n {criterion}: {all_stats[best_fit_line][criterion][0]:.4f}", xy = (0.6, 0.15), xycoords = "axes fraction", fontsize = 10,)
        ax.legend()
        plt.savefig(f"plots/regression/evaluation/{sensor_code}_{class_code}_{mat}_best_fit_curve_{criterion}.svg")
        plt.show()
    
    return best_fit_params, all_curve_params, best_fit_line, all_stats

def invert_predicted_values(predicted_values, y_test, fitted_curves):
    """ 
    Invert the best fitted curves to back-calculate the expected/adjusted FPC values for any given regression output.
    
    """
    predicted_covers = pd.DataFrame(index = y_test.index, columns = y_test.columns)
    for col in range(y_test.shape[1]):
        class_name = y_test.columns[col]
        
        if len(fitted_curves[class_name]) == 4:
            # Access sigmoid params
            L, x0, k, b = fitted_curves[class_name]
            # values = np.array([x if x > b else b for x in predicted_values[:,col] ])
            # Calculate inverse sigmoid for predicted values
            x_result = inverse_sigmoid(predicted_values[:, col], L, x0, k, b)
            x_result = np.clip(x_result, 0, 1)
            
            # Store new predicted values
            predicted_covers[class_name] = x_result
            
        elif len(fitted_curves[class_name]) == 2:
            # Access linear params
            b, m = fitted_curves[class_name]
            # Calculate inverse linear for predicted values
            x_result = (predicted_values[:, col] - b) / m
            x_result = np.clip(x_result, 0, 1)
            
            # Store new predicted values
            predicted_covers[class_name] = x_result
            
    return predicted_covers

def inverse_sigmoid(y, L, x0, k, b):
    """
    Inverse of sigmoid function.
    Given y, returns x.
    """
    
    y = np.asarray(y, dtype=float)

    # Check if y is in valid range
    if np.any((y - b) <= 0) or np.any((y - b) >= L):
        print("Warning: y values outside valid range (b < y < L+b)")
        
    bad = (y - b <= 0) | (y - b >= L)
    if np.any(bad):
        print("Adjusting bad y values:", y[bad])
        
    # Small epsilon to avoid log(0) or negative arguments
    eps = 1e-12

    # Compute the valid range for y
    y_min = b + eps
    y_max = L + b - eps

    # Clip y to the range
    y = np.clip(y, y_min, y_max)
        
    x = x0 - (1/k) * np.log(L/(y - b) - 1)

    return x

def plot_mixpix_components(pixel_fpcs, colours,  sensor_code, class_code, save = False, output_dir = None):
    """ 
    Plot the fractional percent cover distributions of the simulated pixels.
    
    """
    if pixel_fpcs.empty:
        ValueError("No valid pixel fractional percent cover data to plot.")
    
    fpcs = pixel_fpcs.copy()
    fpcs["count"] = fpcs.astype(bool).sum(axis=1)

    mat_count = fpcs.shape[1]
    fig, ax = plt.subplots(mat_count//2 + mat_count%2, 2, figsize = (8,6))
    fig.tight_layout(pad = 3)

    for col in range(mat_count):
        ax[col//2, col%2].hist(fpcs.iloc[:,col], 
                               bins = 50, 
                               range = (0, max(fpcs.iloc[:, col])), 
                               color = colours.get(fpcs.columns[col], "navy"), log = True)
        annotation = fpcs.columns[col].replace("_", " ").capitalize()
        ax[col//2, col%2].annotate(annotation,  
                                   xy = (0.2, 0.8), 
                                   xycoords = "axes fraction")
        
        if fpcs.columns[col] != "count":
            ax[col//2, col%2].set_xlabel("Fractional Cover")
        else:
            ax[col//2, col%2].set_xlabel("Number of Classes Simulated")
            
        ax[col//2, col%2].set_ylabel("Pixel Count")

    if mat_count % 2 == 1:
        ax[-1, -1].set_visible(False)
    fig.suptitle("Distribution of Simulated Fractional Covers", y = 1.02)
    if save: 
        if not sensor_code or not class_code:
            ValueError("Sensor code and class code must be provided to save the figure.")
        if output_dir is None :
            output_dir = "./"
        
        os.makedirs(output_dir, exist_ok=True)
        filename = f"{sensor_code}_{class_code}_mixpix_comps_dist.svg"
        filepath = os.path.join(output_dir, filename)
        fig.savefig(filepath, bbox_inches = "tight")
    plt.show()

In [5]:
# Set persistent notebook parameters
colours = {
    "kelp": "gold",
    "brown_algae": "sienna" ,
    "red_veg": "tomato",
    "green_veg": "green",
    "mineral" : "grey",
    "water" : "teal",
}
sensor_code = "S2"
class_code = "kbrgm"
plot_output_dir = "C:/Users/s4770224/Documents/Work/Writing/Figures/Obj1/part2/"


# mixing pixels

In [ ]:
material_options = {
    # "kelp" : ['ecklonia', 'macrocystis', 'undaria'],
    # "brown_non_kelp" : ['acrocarpia', 'cystophora', 'carpophyllum', 
    #                 'durvillaea', 'hormosira', 'petalonia','phyllospora', 'sargassum', 'scytosiphon'],
    "brown_algae" : ['ecklonia', 'macrocystis', 'petalonia', 
                     'undaria','acrocarpia', 'cystophora', 'carpophyllum', 'durvillaea', 'hormosira', 'phyllospora', 'sargassum', 'scytosiphon'],
    "red_veg" : ['frondose_rhodophyte','filamentous_rhodophyte'], 
    "green_veg" : ["ulva"], #['grass', 'ulva'],
    # "non_brown_autos" : ['grass', 'filamentous_rhodophyte',
    #                      'frondose_rhodophyte', 'rhodophyte', 'ulva'],
    # "all_non_kelp_autos" : ['acrocarpia', 'carpophyllum', 'cystophora', 
    #                         'durvillaea', 'filamentous_rhodophyte','frondose_rhodophyte', 'petalonia', 'grass','hormosira','phyllospora', 'rhodophyte',  'sargassum','scytosiphon','ulva'],
    # "all_non_kelp" : ['acrocarpia', 'barnacle_shells', 'carpophyllum',
    #                   'cystophora', 'durvillaea', 'filamentous_rhodophyte','frondose_rhodophyte', 'gravel', 'grass','hormosira','mussels','petalonia','phyllospora', 'rhodophyte', 'rock','sand', 'sargassum', 'scytosiphon', 'shell_litter','rock', 'ulva', 'worm_castings'],
    "mineral" : ['barnacle_shells', 'gravel', 'mussels', 'rock', 'sand', 
                  'shell_litter', 'worm_castings'],
    }

all_allowed = []
for item in material_options.items():
    all_allowed += item[1]
    
print(f"Classes to be simulated : {material_options.keys()} " )

In [7]:
# #Assign fractional cover ranges with simulate within

cover_ranges = {
    # "kelp" : (0,1),
    # "brown_non_kelp" : (0,1),
    "brown_algae" : (0, 1),
    "red_veg" : (0,1),
    "green_veg" : (0, 1),
    #"non_brown_autos" : (0, 40),
    #"all_non_kelp_autos" : (0, 100),
    #"all_non_kelp" : (0,100),
    "mineral" : (0, 1),}

presence_odds = {
    # "kelp" : (0,1),
    # "brown_non_kelp" : (0,1),
    "brown_algae" : (1, 0),
    "red_veg" : (1, 0),
    "green_veg" : (1, 0),
    #"non_brown_autos" : (0, 40),
    #"all_non_kelp_autos" : (0, 100),
    #"all_non_kelp" : (0,100),
    "mineral" : (1, 0),}

In [8]:
# Load spectral data from file and prep

data = pd.read_csv("./data/processed/resampled/noisy_Sentinel_2_ABC_resampled.csv", index_col=0)
labels = pd.read_csv("data/labels_prepped.csv", index_col=0)
data = pd.concat([labels["Class"], data], axis = 1)
data = data [data["Class"].isin(all_allowed)]
labels = labels[labels["Class"].isin(all_allowed)]

In [ ]:
# Load and prepare water pixel spectra from file

# Load deep water
deep_waters = pd.read_csv("./S2_deep_water_agg.csv", index_col=0)
deep_waters["depth"] = "deep"

# Load shallow water
shallow_waters = pd.read_csv("./S2_shallow_agg.csv", index_col=0)
shallow_waters["depth"] = "shallow"

# combine all water pixels and separate into metadata and spectra
all_waters = pd.concat([deep_waters, shallow_waters], 
                   axis = 0, ignore_index= True).reset_index(drop = True)
water_spec = all_waters.iloc[:, 4:-4]
water_meta = pd.concat([all_waters.iloc[:, :4], all_waters.iloc[:, -4:]], axis = 1)
print(f"Total of {water_spec.shape[0]} water pixels loaded")

# De-duplicate water spectra
water_spec.drop_duplicates(inplace = True)
water_meta = water_meta.loc[water_spec.index]
print(f"Total of {water_spec.shape[0]} unique water pixels")

# Scale water reflectance if needed
if sensor_code == "S2":
    water_spec= water_spec/10000        # S2 reflectance scaling


In [ ]:
# Examine the water spectra being used

clusters = KMeans(n_clusters = 8, n_init ='auto').fit(water_spec)
water_meta["cluster"] = clusters.labels_
water_spec.groupby(water_meta["cluster"]).mean().T.plot()
plt.xticks(rotation = 45)
plt.suptitle("Mean Water Spectra by KMeans Cluster")
plt.ylabel("Log (Reflectance)")
plt.yscale("log")
plt.legend(title = "Cluster", loc = "best")
plt.show()

In [ ]:
# plot mean endmember spectra 

labels["brgm"] = labels["kbrgm"].replace(to_replace=["kelp", "other_brown_alg"], value = "browns")
endmembers = pd.DataFrame(data.iloc[:, 1:]).groupby(labels["brgm"]).mean()

plt.plot(endmembers.T)
plt.legend(endmembers.index)
plt.xticks(rotation = 45)
plt.ylabel("Reflectance")
plt.xlabel("Band")  
plt.show()

In [ ]:
#plot kmeans clusters
plt.scatter((water_spec.iloc[:, 1]-water_spec.iloc[:, 2]), water_spec.iloc[:, 3], c = water_meta["cluster"], cmap = "tab20", alpha = 0.4)
plt.yscale("log")
plt.ylabel("Red Edge reflectance")
plt.xlabel( "Normalized green and red index")
plt.title("Water clusters")
plt.show()

In [13]:
# Split spectra into training and testing sets
data_train, data_test, labels_train, labels_test = train_test_split(data, labels, test_size = 0.3, stratify=labels["kbrgm"])

water_train, water_test = train_test_split(water_spec, test_size = 0.3)
water_meta_train = water_meta.loc[water_train.index]
water_meta_test = water_meta.loc[water_test.index]

In [ ]:
# Instantiate the pixel mixer
mixer = lmx.LinearMixing(materials = material_options, 
                         cover_ranges = cover_ranges, 
                         pixels = 10000,
                         odds_dict = presence_odds,
                         material_spectra= data_train, 
                         water_spectra = water_train)

# Mix pixels and store results
results, f, endmember_indices = mixer.sim_many_pixels()

# Format results and clear old variables
sim_pix, pixel_fpcs, endmember_indices = mixer.format_sim_results(results, f, endmember_indices)
del results, f

# Inspect simulated results
print(f"{sim_pix.shape[0]} simulated pixels with {sim_pix.shape[1]} spectral bands \n")
print(f"Preview of simulated component fractional covers: \n {pixel_fpcs.head()}")


In [ ]:
# Plot the components of the simulated pixels      
plot_mixpix_components(pixel_fpcs, colours, sensor_code= sensor_code, class_code = class_code, save = True, output_dir = plot_output_dir)

In [16]:
# Export simulated pixels

#simulated_pixels.to_csv(f"data/mixed_sims/{sensor_code}_{class_code}_mixpix.csv")
#pixel_fpcs.to_csv(f"data/mixed_sims/{sensor_code}_{class_code}_fpcs.csv)

# Prepare data for classification

In [17]:
# Define what you simulated
label_level = "kbrgm"
keep_list = ["kelp", "water", "other_brown_alg", "mineral", "green_veg", "red_veg"]

# Add water to classifier training set 
sample_idx = np.random.choice(water_train.index, size = 1000, replace = False)
water_labels = pd.DataFrame("water", index = sample_idx, columns = labels.columns)
water_sample = water_train.loc[sample_idx, :]
water_sample.columns = data_train.columns[1:]
data_watered = pd.concat([data_train, water_sample], axis = 0)
labels_watered = pd.concat([labels_train, water_labels], axis = 0).set_index(data_watered.index)
data_watered = pd.concat([labels_watered, data_watered.iloc[:, 1:]], axis = 1)


# Combine kelp and other browns into cingle class
labels_watered["brgm"] = labels_watered["kbrgm"].replace(to_replace=["kelp", "brown_non_kelp"], value = "browns")

# Filter training data to only mixed pixel components
data_watered = data_watered[data_watered[label_level].isin(keep_list)]
water_endmember = data_watered[label_level] == "water"


In [20]:
# Extract PCA components and transform dataset
deco = PCA(min(sim_pix.shape[1], 15))
deco.fit(sim_pix) #.iloc[:,8:]) #ENDMEMBERS?
# decomposed_endmembers = deco.transform(data_watered.iloc[:, 8:])
decomposed_mixpix = deco.transform(sim_pix)
comps = deco.components_

In [ ]:
#plot the components

plt.plot(comps[:min(sim_pix.shape[1], 10), :].T, label = range(min(sim_pix.shape[1], 10)))
plt.legend(title = "Component")
plt.xlabel("Band")
plt.ylabel("Component weight")
plt.show()

In [22]:
pixel_fpcs["maj"] = pixel_fpcs.iloc[:, :-1].idxmax(axis = 1)
fpcs = pixel_fpcs.iloc[:, :-1]

# Perform presence Absence classification

In [24]:
# divide the dataset into training and testing
x_train, x_test, y_train, y_test = train_test_split(decomposed_mixpix, fpcs, train_size = 0.5, random_state = 7)
x_train = pd.DataFrame(x_train, index= y_train.index)
x_test = pd.DataFrame(x_test, index = y_test.index)

In [25]:
rf = RandomForestRegressor(
    bootstrap = True,
    ccp_alpha= 0.0,
    criterion = 'absolute_error',
    n_jobs = -1,
    oob_score = True,
    random_state = 123,
    verbose = 1,
    warm_start = False,
    max_features = "sqrt",
    )

In [48]:
classifier_params = {
    "n_estimators": 200,
    "max_depth": 25,
    "min_samples_split": 7,
    "min_samples_leaf": 2,
}

In [ ]:
pipe = rfcc.Unmix_Pixels(x_train, y_train, x_test, y_test, presence_absence_thresh= 0.2)
presence_probas = pipe.classify_and_append(classifier_params, plot = "water")

In [50]:
# Store classifications and water reg results
train_classifications = pd.DataFrame({k : v for k, v in presence_probas["train"].items()})
test_classifications = pd.DataFrame({k : v for k, v in presence_probas["test"].items()})


In [ ]:
# Plot soft classification results
plotting_params = {
    "x_array": y_test, 
    "y_array": test_classifications, 
    "alpha": 0.4, # p[:,-1],
    "metric" : "MAE", 
    "colour": "grey", #p[:,-1],
    "cmap" : "viridis", 
    "fig_title" : "Soft classification - probability of class presence",
    "x_axis_label": "Simulated fractional cover",
    "y_axis_label" : "Probability of class presence", 
    "stats_values" : None,
    "line_1_1" : False, 
    "sensor_code" : sensor_code,
    "class_code" : class_code,
    "h_lines" : [(0.2, "gold"), (0.5, "red")],
    "save" : True,
    "output_dir" : plot_output_dir,
    "output_name" : "rf_chain_soft_classification_testset"
}

plot_multi_result(**plotting_params)  

In [33]:
# Change regressor hyperparams if desired
regressor_params = {
    'bootstrap': True,
    'ccp_alpha': 0.0,
    'criterion': 'absolute_error',
    'max_depth': 10,
    'max_features': 'sqrt',
    'max_leaf_nodes': None,
    'max_samples': 0.6,
    'min_impurity_decrease': 0.0,
    'min_samples_leaf': 5,
    'min_samples_split': 15,
    'min_weight_fraction_leaf': 0.0,
    'monotonic_cst': None,
    'n_estimators': 100,
    'n_jobs': -1,
    'oob_score': True,
    'random_state': 123,
    'verbose': 0,
    'warm_start': False,  
}

In [ ]:
regressor_predictions = pipe.filter_and_regress(plot = "none", regressor_params= regressor_params)

In [ ]:
# Store regression results
train_regressions = pd.DataFrame({k : v.iloc[:, 0] for k, v in regressor_predictions["train"].items()})
test_regressions = pd.DataFrame({k : v.iloc[:, 0] for k, v in regressor_predictions["test"].items()})


# Combine regression results and classifications based on presence threshold
overall_train_results = pd.DataFrame(index = train_classifications.index, columns= train_classifications.columns)
for col in train_regressions.columns:
    for row in train_regressions.index:
        if train_classifications.loc[row, col] > 0.2:
            overall_train_results.loc[row, col] = train_regressions.loc[row, col]
        else:
            overall_train_results.loc[row, col] = 0
overall_train_results["water"] = train_classifications["water"]

overall_test_results = pd.DataFrame(index = test_classifications.index, columns= test_classifications.columns)
for col in test_regressions.columns:
    for row in test_regressions.index:
        if test_classifications.loc[row, col] > 0.2:
            overall_test_results.loc[row, col] = test_regressions.loc[row, col]
        else:
            overall_test_results.loc[row, col] = 0
overall_test_results["water"] = test_classifications["water"]



# Adjust for unity
overall_test_area_adjusted = overall_test_results.div(overall_test_results.sum(axis = 1), axis = 0)

overall_train_results.replace(np.nan, 0, inplace = True)
overall_test_results.replace(np.nan, 0, inplace = True)
overall_test_area_adjusted.replace(np.nan, 0, inplace = True)



# Calculate evaluation stats
overall_train_stats = calc_eval_stats(y_train, overall_train_results)
overall_test_stats = calc_eval_stats(y_test, overall_test_results)
area_adjusted_stats = calc_eval_stats(y_test, overall_test_area_adjusted)

In [ ]:
# Plot regressor results

plotting_params = {
    "x_array": y_train, 
    "y_array": overall_train_results, 
    "alpha": 0.4, # p[:,-1],
    "metric" : "MAE", 
    "colour":overall_train_results.iloc[:, 0], #p[:,-1],
    "cmap" : "viridis", 
    "fig_title" : "Regression results - Train SET",
    "x_axis_label": "Fractional cover",
    "y_axis_label" : "Predicted cover", 
    "stats_values" : overall_train_stats,
    "line_1_1" : True, 
    "sensor_code" : sensor_code,
    "class_code" : class_code,
    "save" : True,
    "output_dir" : "./plots/regression/evaluation/", 
    "output_name": "reg_test_water_fpc"
}

plot_multi_result(**plotting_params)  

In [ ]:
# Plot regressor results

plotting_params = {
    "x_array": y_test, 
    "y_array": overall_test_results, 
    "alpha": 0.4, # p[:,-1],
    "metric" : "MAE", 
    "colour":overall_test_results.iloc[:, -1], #p[:,-1],
    "cmap" : "viridis", 
    "fig_title" : "Regression results - TEST SET",
    "x_axis_label": "Fractional cover",
    "y_axis_label" : "Predicted cover", 
    "stats_values" : overall_test_stats,
    "line_1_1" : True, 
    "sensor_code" : sensor_code,
    "class_code" : class_code,
    "save" : True,
    "output_dir" : "./plots/regression/evaluation/", 
    "output_name": "reg_test_water_fpc"
}

plot_multi_result(**plotting_params)  

In [ ]:
# Plot adjusted regressor results

plotting_params = {
    "x_array": y_test, 
    "y_array": overall_test_area_adjusted, 
    "alpha": 0.4, # p[:,-1],
    "metric" : "MAE", 
    "colour":overall_test_area_adjusted.iloc[:, -1], #p[:,-1],
    "cmap" : "viridis", 
    "fig_title" : "Adjusted random forest results",
    "x_axis_label": "Fractional cover",
    "y_axis_label" : "Predicted cover", 
    "stats_values" : area_adjusted_stats,
    "line_1_1" : True, 
    "sensor_code" : sensor_code,
    "class_code" : class_code,
    "save" : True,
    "output_dir" : "./plots/regression/evaluation/", 
    "output_name": "reg_test_adjusted_water_fpc"
}

plot_multi_result(**plotting_params)  

# Fitting curves and inverting

In [39]:
rfr = RandomForestRegressor(**regressor_params)
rfr.fit(x_train, y_train)
train_pred = rfr.predict(x_train)
test_pred = rfr.predict(x_test)


In [ ]:
fitted_curves = {}
curve_stats = {}
for col in range(y_test.shape[1]):
    y_non_zero = y_test[y_test.iloc[:, col] > 0 ]
    p_non_zero = test_pred[y_test.iloc[:,col] >0]
    colour = colours.get(y_test.columns[col], "magenta")
    best_fit_params, all_curve_params, best_fit_line, all_stats = fit_best_curve(
        y_non_zero.iloc[:, col], 
        p_non_zero[:, col], 
        y_non_zero.columns[col], 
        plot = True, 
        criterion = "MAE", 
        colours = colour, 
        sensor_code = sensor_code, 
        class_code = class_code)
    print(f"Best fit curve is a {best_fit_line.capitalize()} for {y_test.columns[col]} with parameters {best_fit_params}")
    fitted_curves[y_test.columns[col]] = best_fit_params
    curve_stats[y_test.columns[col]] = all_stats
    
del y_non_zero, p_non_zero


In [ ]:
# Invert and adjust for totality
adjusted_predicted = invert_predicted_values(test_pred, y_test, fitted_curves)
area_adjusted = adjusted_predicted.div(adjusted_predicted.sum(axis = 1), axis = 0)
adj_stats = calc_eval_stats(y_test, area_adjusted)

In [ ]:

plotting_params = {
    "x_array": y_test, 
    "y_array": area_adjusted, 
    "alpha": 0.3, #y_test.iloc[:,-1],
    "colour": y_test.iloc[:,-1],
    "metric" : "MAE",
    "cmap" : "viridis", 
    "fig_title" : "Regression results on adjusted, inverted testing set",
    "x_axis_label": "fractional cover",
    "y_axis_label" : "Predicted cover", 
    "stats_values" : adj_stats, 
    "line_1_1" : True,
    "sensor_code" : sensor_code,
    "class_code" : class_code, 
    "save" : False,
    "output_dir" : "./plots/regression/evaluation/", 
    "output_name": "adj_reg_test_water_fpc"
}
plot_multi_result(**plotting_params)

# Run Chained classification-regression on new data

In [ ]:
# Instantiate the pixel mixer
mixer = lmx.LinearMixing(materials = material_options, 
                         cover_ranges = cover_ranges, 
                         pixels = 2000, 
                         material_spectra= data_test,
                         odds_dict= presence_odds, 
                         water_spectra = water_test)

# Mix pixels and store results
new_results, new_f, new_endmember_indices = mixer.sim_many_pixels()

# Format results and clear old variables
new_mixpix, new_pixel_fpcs, new_endmembers = mixer.format_sim_results(new_results, new_f, new_endmember_indices)
del new_results, new_f, new_endmember_indices


# Inspect simulated results
print(f"{new_mixpix.shape[0]} pixels simulated with {new_mixpix.shape[1]} spectral bands \n")
print(new_pixel_fpcs.head())

In [62]:
deco_new_mixpix = pd.DataFrame(deco.transform(new_mixpix))

In [69]:
# Unmix using the classifier and regressor pipeline
new_pixel_classifications, new_pixel_regressions = pipe.unmix_new_data(deco_new_mixpix, pd.DataFrame(new_pixel_fpcs))

In [70]:
# Format the output results
new_pixel_classifications = pd.DataFrame({k : v.iloc[:,0] for k, v in new_pixel_classifications.items()})
new_pixel_regressions = pd.DataFrame({k : v.iloc[:, 0] for k, v in new_pixel_regressions.items()})

In [71]:
# Make any pixel not classified as present have zero cover

new_overall_test_results = pd.DataFrame(index = new_pixel_classifications.index, columns= new_pixel_classifications.columns)
for col in new_pixel_regressions.columns:
    for row in new_pixel_regressions.index:
        if new_pixel_classifications.loc[row, col] > 0.2:
            new_overall_test_results.loc[row, col] = new_pixel_regressions.loc[row, col]
        else:
            new_overall_test_results.loc[row, col] = 0
new_overall_test_results["water"] = new_pixel_classifications["water"]

In [ ]:
# Calculate evaluation stats
new_stats = calc_eval_stats(new_pixel_fpcs, new_overall_test_results)
new_stats["MAE"]

In [ ]:
# Plot new data regression results
plotting_params = {
    "x_array": new_pixel_fpcs, 
    "y_array": new_overall_test_results, 
    "alpha": 0.7, #new_overall_test_results.iloc[:,-1], 
    "colour": new_endmembers.iloc[:, 1], #new_overall_test_results.iloc[:,-1], #new_endmembers.iloc[:, 4],
    "metric" : "MAE",
    "cmap" : "tab20", 
    "fig_title" : "Adjusted random forest regressor results",
    "x_axis_label": "Fractional cover",
    "y_axis_label" : "Predicted cover", 
    "stats_values" : new_stats,
    "line_1_1" : True,
    "sensor_code" : sensor_code,
    "class_code" : class_code, 
    "save" : False,
    "output_dir" : "./plots/regression/evaluation/", 
    "output_name": "reg_new_data_mineral_endmember"
}
plot_multi_result(**plotting_params)  

In [ ]:
# Plot the endmembers of a chosen class
cleaned = [x for x in new_endmembers.iloc[:, 3].unique() if not isnan(x)]

plt.plot(data_test.loc[cleaned, 'S2A_SR_AV_B2':'S2A_SR_AV_B8A'].T)
plt.legend(cleaned)
plt.xticks(rotation = 45)
plt.xlabel("Sensor Band")
plt.ylabel("Reflectance")
plt.show()

In [ ]:
# Invert and adjust for totality
new_adjusted_predicted = invert_predicted_values(np.array(new_overall_test_results), new_pixel_fpcs, fitted_curves)
new_area_adjusted = new_adjusted_predicted.div(new_adjusted_predicted.sum(axis = 1), axis = 0)
new_adj_inv_stats = calc_eval_stats(new_pixel_fpcs, new_area_adjusted)

In [103]:
new_adj_stats = calc_eval_stats(new_pixel_fpcs, new_overall_test_results.div(new_overall_test_results.sum(axis = 1), axis = 0))

In [ ]:
print(f"Inverted and Area adjusted: \n {pd.DataFrame(new_adj_inv_stats)}\n \n")
print(f"Regression predictions : \n {pd.DataFrame(new_stats)} \n \n ")
print(f"Area adjusted regression predictions : \n {pd.DataFrame(new_adj_stats)}")

In [ ]:
# Plot regression results coloured by class endmember
plotting_params = {
    "x_array": new_pixel_fpcs, 
    "y_array": new_area_adjusted, 
    "alpha": 0.3, #new_p[:,-1],
    "colour": new_endmembers.iloc[:, 4], # new_p[:,-1],
    "metric": "MAE",
    "cmap" : "viridis", 
    "fig_title" : "NEW DATA - Adjusted, inverted regression results",
    "x_axis_label": "Fractional cover",
    "y_axis_label" : "Predicted cover", 
    "stats_values" : new_adj_inv_stats,
    "line_1_1" : True, 
    "sensor_code" : sensor_code,
    "class_code" : class_code, 
    "save" : True,
    "output_dir" : "./plots/regression/evaluation/", 
    "output_name": "adj_reg_new_data_water_endmembers"
}
plot_multi_result(**plotting_params)  

In [ ]:
# Plot regression results coloured by class endmember
plotting_params = {
    "x_array": new_pixel_fpcs, 
    "y_array": new_overall_test_results, 
    "alpha": 0.3, #new_p[:,-1],
    "colour": new_endmembers.iloc[:, 4], # new_p[:,-1],
    "metric": "MAE",
    "cmap" : "viridis", 
    "fig_title" : "NEW DATA - Adjusted regression results",
    "x_axis_label": "Fractional cover",
    "y_axis_label" : "Predicted cover", 
    "stats_values" : new_adj_stats,
    "line_1_1" : True, 
    "sensor_code" : sensor_code,
    "class_code" : class_code, 
    "save" : True,
    "output_dir" : "./plots/regression/evaluation/", 
    "output_name": "adj_reg_new_data_water_endmembers"
}
plot_multi_result(**plotting_params)  